In [ ]:
# 02 5 km Land-Cover Fraction Grid

## Purpose
Convert the ESA WorldCover raster into an analysis-ready 5 km deployment-site table.

## Inputs
- `data/processed/nigeria_worldcover_2021_mosaic.tif`
- Nigeria ADM0 boundary
- 5 km grid generated in Python

## Main processing steps
- Generate 5 km grid cells covering Nigeria
- Assign a unique `cell_id` to each grid cell
- Calculate land-cover fractions for each grid cell
- Validate that fractions sum to 1
- Export the land-cover deployment-site table

## Outputs
- `outputs/tables/nigeria_landcover_fraction_5km.csv`
- `outputs/tables/nigeria_landcover_fraction_5km_sample500.csv`

In [2]:
from pathlib import Path
import rasterio
from rasterio.mask import mask
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import box
from tqdm import tqdm

print("Libraries loaded.")

Libraries loaded.


In [3]:
mosaic_path = Path("../data/processed/nigeria_worldcover_2021_mosaic.tif")
boundary_path = Path("../data/raw/boundaries/geoBoundaries-NGA-ADM0_simplified.geojson")

print("Mosaic exists:", mosaic_path.exists())
print("Boundary exists:", boundary_path.exists())

Mosaic exists: True
Boundary exists: True


In [4]:
nigeria = gpd.read_file(boundary_path)

print("Original CRS:", nigeria.crs)
nigeria.head()

Original CRS: EPSG:4326


,shapeName,shapeISO,shapeID,shapeGroup,shapeType,geometry
0,the Federal Republic of Nigeria,NGA,39225661B33378512450279,NGA,ADM0,"POLYGON ((5.67705 13.82055, 5.63161 13.83633, ..."


In [5]:
# Project Nigeria to a meter-based CRS
nigeria_3857 = nigeria.to_crs("EPSG:3857")

minx, miny, maxx, maxy = nigeria_3857.total_bounds

grid_size = 5000  # 5 km

cells = []
cell_ids = []

cell_id = 0

x = minx
while x < maxx:
    y = miny
    while y < maxy:
        grid_cell = box(x, y, x + grid_size, y + grid_size)

        # Keep only cells that intersect Nigeria
        if nigeria_3857.intersects(grid_cell).any():
            cells.append(grid_cell)
            cell_ids.append(f"NGA_5km_{cell_id:06d}")
            cell_id += 1

        y += grid_size
    x += grid_size

grid_5km = gpd.GeoDataFrame(
    {"cell_id": cell_ids},
    geometry=cells,
    crs="EPSG:3857"
)

print("Number of 5 km grid cells:", len(grid_5km))
grid_5km.head()

Number of 5 km grid cells: 38458


,cell_id,geometry
0,NGA_5km_000000,"POLYGON ((304740.257 710797.614, 304740.257 71..."
1,NGA_5km_000001,"POLYGON ((304740.257 715797.614, 304740.257 72..."
2,NGA_5km_000002,"POLYGON ((304740.257 720797.614, 304740.257 72..."
3,NGA_5km_000003,"POLYGON ((304740.257 725797.614, 304740.257 73..."
4,NGA_5km_000004,"POLYGON ((304740.257 730797.614, 304740.257 73..."


In [6]:
grid_centroids = grid_5km.copy()
grid_centroids["geometry"] = grid_centroids.geometry.centroid
grid_centroids = grid_centroids.to_crs("EPSG:4326")

grid_5km["longitude"] = grid_centroids.geometry.x
grid_5km["latitude"] = grid_centroids.geometry.y

grid_5km.head()

,cell_id,geometry,longitude,latitude
0,NGA_5km_000000,"POLYGON ((304740.257 710797.614, 304740.257 71...",2.71507,6.394346
1,NGA_5km_000001,"POLYGON ((304740.257 715797.614, 304740.257 72...",2.71507,6.438981
2,NGA_5km_000002,"POLYGON ((304740.257 720797.614, 304740.257 72...",2.71507,6.483611
3,NGA_5km_000003,"POLYGON ((304740.257 725797.614, 304740.257 73...",2.71507,6.528238
4,NGA_5km_000004,"POLYGON ((304740.257 730797.614, 304740.257 73...",2.71507,6.572860


In [7]:
landcover_classes = {
    10: "trees",
    20: "shrubland",
    30: "grassland",
    40: "cropland",
    50: "built_up",
    60: "bare_sparse_vegetation",
    70: "snow_ice",
    80: "permanent_water_bodies",
    90: "herbaceous_wetland",
    95: "mangroves",
    100: "moss_lichen",
}

print("Classes defined.")

Classes defined.


In [8]:
def calculate_fraction_for_cell(src, cell_geom_3857):
    cell_gdf = gpd.GeoDataFrame(geometry=[cell_geom_3857], crs="EPSG:3857")
    cell_gdf = cell_gdf.to_crs(src.crs)

    try:
        out_image, _ = mask(
            src,
            cell_gdf.geometry,
            crop=True,
            nodata=0
        )

        data = out_image[0]
        valid = data[data != 0]

        total_pixels = len(valid)

        fractions = {}

        if total_pixels == 0:
            for class_name in landcover_classes.values():
                fractions[class_name] = 0
            fractions["fraction_sum"] = 0
            fractions["valid_pixel_count"] = 0
            return fractions

        for code, class_name in landcover_classes.items():
            fractions[class_name] = np.sum(valid == code) / total_pixels

        fractions["fraction_sum"] = sum(
            fractions[class_name] for class_name in landcover_classes.values()
        )
        fractions["valid_pixel_count"] = total_pixels

        return fractions

    except Exception:
        fractions = {class_name: 0 for class_name in landcover_classes.values()}
        fractions["fraction_sum"] = 0
        fractions["valid_pixel_count"] = 0
        return fractions

In [9]:
test_rows = []

with rasterio.open(mosaic_path) as src:
    for idx, row in tqdm(grid_5km.head(50).iterrows(), total=50):
        fractions = calculate_fraction_for_cell(src, row.geometry)

        result = {
            "cell_id": row["cell_id"],
            "longitude": row["longitude"],
            "latitude": row["latitude"],
        }

        result.update(fractions)
        test_rows.append(result)

test_df = pd.DataFrame(test_rows)

test_df.head()

100%|██████████| 50/50 [00:16<00:00,  3.03it/s]


,cell_id,longitude,latitude,trees,shrubland,grassland,cropland,built_up,bare_sparse_vegetation,snow_ice,permanent_water_bodies,herbaceous_wetland,mangroves,moss_lichen,fraction_sum,valid_pixel_count
0,NGA_5km_000000,2.71507,6.394346,0.338479,0.020117,0.201586,0.004714,0.244382,0.011917,0.0,0.091124,0.087680,0.000000,0.0,1.0,288904
1,NGA_5km_000001,2.71507,6.438981,0.407654,0.009360,0.134719,0.001627,0.057950,0.000177,0.0,0.167329,0.220170,0.001014,0.0,1.0,288904
2,NGA_5km_000002,2.71507,6.483611,0.898153,0.000843,0.037976,0.005157,0.053852,0.000541,0.0,0.000000,0.003478,0.000000,0.0,1.0,288365
3,NGA_5km_000003,2.71507,6.528238,0.835599,0.032848,0.045264,0.011118,0.056472,0.000000,0.0,0.000000,0.018698,0.000000,0.0,1.0,288904
4,NGA_5km_000004,2.71507,6.572860,0.823949,0.025523,0.037824,0.044582,0.064980,0.000003,0.0,0.000000,0.003138,0.000000,0.0,1.0,288365


In [10]:
test_df[["cell_id", "fraction_sum", "valid_pixel_count"]].head()

,cell_id,fraction_sum,valid_pixel_count
0,NGA_5km_000000,1.0,288904
1,NGA_5km_000001,1.0,288904
2,NGA_5km_000002,1.0,288365
3,NGA_5km_000003,1.0,288904
4,NGA_5km_000004,1.0,288365


In [11]:
grid_5km

,cell_id,geometry,longitude,latitude
0,NGA_5km_000000,"POLYGON ((304740.257 710797.614, 304740.257 71...",2.715070,6.394346
1,NGA_5km_000001,"POLYGON ((304740.257 715797.614, 304740.257 72...",2.715070,6.438981
2,NGA_5km_000002,"POLYGON ((304740.257 720797.614, 304740.257 72...",2.715070,6.483611
3,NGA_5km_000003,"POLYGON ((304740.257 725797.614, 304740.257 73...",2.715070,6.528238
4,NGA_5km_000004,"POLYGON ((304740.257 730797.614, 304740.257 73...",2.715070,6.572860
...,...,...,...,...
38453,NGA_5km_038453,"POLYGON ((1634740.257 1340797.614, 1634740.257...",14.662664,11.978816
38454,NGA_5km_038454,"POLYGON ((1634740.257 1350797.614, 1634740.257...",14.662664,12.066677
38455,NGA_5km_038455,"POLYGON ((1634740.257 1355797.614, 1634740.257...",14.662664,12.110597
38456,NGA_5km_038456,"POLYGON ((1634740.257 1360797.614, 1634740.257...",14.662664,12.154509


In [12]:
sample_grid = grid_5km.sample(500, random_state=42)

sample_rows = []

with rasterio.open(mosaic_path) as src:
    for idx, row in tqdm(sample_grid.iterrows(), total=len(sample_grid)):
        fractions = calculate_fraction_for_cell(src, row.geometry)

        result = {
            "cell_id": row["cell_id"],
            "longitude": row["longitude"],
            "latitude": row["latitude"],
        }

        result.update(fractions)
        sample_rows.append(result)

df_sample_500 = pd.DataFrame(sample_rows)

df_sample_500.head()

100%|██████████| 500/500 [01:55<00:00,  4.33it/s]


,cell_id,longitude,latitude,trees,shrubland,grassland,cropland,built_up,bare_sparse_vegetation,snow_ice,permanent_water_bodies,herbaceous_wetland,mangroves,moss_lichen,fraction_sum,valid_pixel_count
0,NGA_5km_021171,8.284625,11.759039,0.000718,0.031596,0.291203,0.619050,0.056292,0.001109,0.0,0.000032,0.000000,0.0,0.0,1.0,284053
1,NGA_5km_005893,4.871027,13.031205,0.009930,0.210287,0.193296,0.557148,0.007382,0.016266,0.0,0.001042,0.004647,0.0,0.0,1.0,282975
2,NGA_5km_032816,11.653307,10.745849,0.000231,0.013033,0.179942,0.786519,0.019698,0.000578,0.0,0.000000,0.000000,0.0,0.0,1.0,285670
3,NGA_5km_027143,9.946508,11.186803,0.000000,0.608938,0.074268,0.315416,0.001294,0.000084,0.0,0.000000,0.000000,0.0,0.0,1.0,285131
4,NGA_5km_021541,8.374457,11.010501,0.003242,0.007047,0.008062,0.967497,0.013285,0.000851,0.0,0.000000,0.000018,0.0,0.0,1.0,285670


In [13]:
df_sample_500[["cell_id", "fraction_sum", "valid_pixel_count"]].describe()

,fraction_sum,valid_pixel_count
count,5.000000e+02,500.000000
mean,1.000000e+00,286179.978000
std,7.219401e-17,1939.751948
min,1.000000e+00,281897.000000
25%,1.000000e+00,284592.000000
50%,1.000000e+00,286209.000000
75%,1.000000e+00,287826.000000
max,1.000000e+00,289982.000000


In [14]:
df_sample_500[
    (df_sample_500["fraction_sum"] < 0.99) |
    (df_sample_500["fraction_sum"] > 1.01) |
    (df_sample_500["valid_pixel_count"] == 0)
]

,cell_id,longitude,latitude,trees,shrubland,grassland,cropland,built_up,bare_sparse_vegetation,snow_ice,permanent_water_bodies,herbaceous_wetland,mangroves,moss_lichen,fraction_sum,valid_pixel_count


In [15]:
sample_output_path = Path("../outputs/tables/nigeria_landcover_fraction_5km_sample500.csv")
sample_output_path.parent.mkdir(parents=True, exist_ok=True)

df_sample_500.to_csv(sample_output_path, index=False)

print(f"Sample saved to: {sample_output_path}")

Sample saved to: ..\outputs\tables\nigeria_landcover_fraction_5km_sample500.csv


In [16]:
all_rows = []

with rasterio.open(mosaic_path) as src:
    for idx, row in tqdm(grid_5km.iterrows(), total=len(grid_5km)):
        fractions = calculate_fraction_for_cell(src, row.geometry)

        result = {
            "cell_id": row["cell_id"],
            "longitude": row["longitude"],
            "latitude": row["latitude"],
        }

        result.update(fractions)
        all_rows.append(result)

df_5km = pd.DataFrame(all_rows)

df_5km.head()

100%|██████████| 38458/38458 [2:22:43<00:00,  4.49it/s]  


,cell_id,longitude,latitude,trees,shrubland,grassland,cropland,built_up,bare_sparse_vegetation,snow_ice,permanent_water_bodies,herbaceous_wetland,mangroves,moss_lichen,fraction_sum,valid_pixel_count
0,NGA_5km_000000,2.71507,6.394346,0.338479,0.020117,0.201586,0.004714,0.244382,0.011917,0.0,0.091124,0.087680,0.000000,0.0,1.0,288904
1,NGA_5km_000001,2.71507,6.438981,0.407654,0.009360,0.134719,0.001627,0.057950,0.000177,0.0,0.167329,0.220170,0.001014,0.0,1.0,288904
2,NGA_5km_000002,2.71507,6.483611,0.898153,0.000843,0.037976,0.005157,0.053852,0.000541,0.0,0.000000,0.003478,0.000000,0.0,1.0,288365
3,NGA_5km_000003,2.71507,6.528238,0.835599,0.032848,0.045264,0.011118,0.056472,0.000000,0.0,0.000000,0.018698,0.000000,0.0,1.0,288904
4,NGA_5km_000004,2.71507,6.572860,0.823949,0.025523,0.037824,0.044582,0.064980,0.000003,0.0,0.000000,0.003138,0.000000,0.0,1.0,288365


In [17]:
df_5km[["fraction_sum", "valid_pixel_count"]].describe()

,fraction_sum,valid_pixel_count
count,3.845800e+04,38458.000000
mean,1.000000e+00,286205.253211
std,7.000913e-17,1943.389231
min,1.000000e+00,281374.000000
25%,1.000000e+00,284592.000000
50%,1.000000e+00,286209.000000
75%,1.000000e+00,287826.000000
max,1.000000e+00,289982.000000


In [18]:
output_path = Path("../outputs/tables/nigeria_landcover_fraction_5km.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_5km.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: ..\outputs\tables\nigeria_landcover_fraction_5km.csv
